<a href="https://colab.research.google.com/github/sayam-h069/Compiler_Design/blob/main/CD_Lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1

In [ ]:
!apt-get update -qq
!apt-get install -y flex gcc

In [2]:
%%writefile vehicle.l
%{
#include <stdio.h>
%}

%%

[A-Z]{2}[0-9]{2}[A-Z]{1,2}[0-9]{4} {
    printf("VALID VEHICLE: %s\n", yytext);
}

[A-Z0-9]+ {
    printf("INVALID: %s\n", yytext);
}

[ \t\n]+ ;

. ;

%%

int yywrap()
{
    return 1;
}

int main()
{
    yylex();
    return 0;
}

Writing vehicle.l


In [5]:
!flex vehicle.l
!gcc lex.yy.c -o vehicle

In [6]:
!echo "TS09EA1234 KA5AB123 AP31AB9876 MH12A1234" | ./vehicle

VALID VEHICLE: TS09EA1234
INVALID: KA5AB123
VALID VEHICLE: AP31AB9876
VALID VEHICLE: MH12A1234


#2

In [7]:
%%writefile source.c
int a = 10;
float b = a + 20;
if(a < b)
{
    return a;
}

Writing source.c


In [8]:
!cat source.c

int a = 10;
float b = a + 20;
if(a < b)
{
    return a;
}


In [9]:
%%writefile analyzer.l
%{
#include <stdio.h>

int keywords = 0;
int identifiers = 0;
int numbers = 0;
int operators = 0;
int special_symbols = 0;
int lines = 0;
%}

%%

"int"|"float"|"if"|"else"|"while"|"return" {
    keywords++;
}

[a-zA-Z][a-zA-Z0-9]* {
    identifiers++;
}

[0-9]+ {
    numbers++;
}

[+\-*/=] {
    operators++;
}

[;,(){}] {
    special_symbols++;
}

\n {
    lines++;
}

[ \t]+ ;

. ;

%%

int yywrap()
{
    return 1;
}

int main()
{
    FILE *fp;

    fp = fopen("source.c", "r");

    if (fp == NULL)
    {
        printf("Error: Cannot open source.c\n");
        return 1;
    }

    yyin = fp;

    yylex();

    fclose(fp);

    printf("\n----- Lexical Statistics -----\n");
    printf("Keywords       : %d\n", keywords);
    printf("Identifiers    : %d\n", identifiers);
    printf("Numbers        : %d\n", numbers);
    printf("Operators      : %d\n", operators);
    printf("Special Symbols: %d\n", special_symbols);
    printf("Lines          : %d\n", lines);

    return 0;
}

Writing analyzer.l


In [10]:
!flex analyzer.l
!gcc lex.yy.c -o analyzer

In [11]:
!./analyzer


----- Lexical Statistics -----
Keywords       : 4
Identifiers    : 6
Numbers        : 2
Operators      : 3
Special Symbols: 7
Lines          : 6


# 3

In [12]:
%%writefile marks.txt
101 CS 85
102 MA 35
10 CS 90
103 EE 120

Writing marks.txt


In [13]:
!cat marks.txt

101 CS 85
102 MA 35
10 CS 90
103 EE 120


In [14]:
%%writefile marks.l
%{
#include <stdio.h>

int total = 0;
int valid = 0;
int invalid = 0;
int passed = 0;
int failed = 0;

char roll[20];
char subject[20];
int marks;
%}

%%

^[0-9]{3}[ \t]+[A-Z]{2}[ \t]+(100|[0-9]{1,2})[ \t]*\n {
    total++;
    valid++;

    sscanf(yytext, "%s %s %d", roll, subject, &marks);

    if (marks >= 40)
    {
        passed++;
        printf("Valid Record - PASS : %s %s %d\n",
               roll, subject, marks);
    }
    else
    {
        failed++;
        printf("Valid Record - FAIL : %s %s %d\n",
               roll, subject, marks);
    }
}

^[0-9]{3}[ \t]+[A-Z]{2}[ \t]+[0-9]+[ \t]*\n {
    total++;
    invalid++;

    sscanf(yytext, "%s %s %d", roll, subject, &marks);

    printf("Invalid Record (Marks out of range) : %s %s %d\n",
           roll, subject, marks);
}

^[^\n]*\n {
    total++;
    invalid++;

    printf("Invalid Record : %.*s\n",
           yyleng - 1, yytext);
}

[ \t]+ ;

\n ;

%%

int yywrap()
{
    return 1;
}

int main()
{
    FILE *fp;

    fp = fopen("marks.txt", "r");

    if (fp == NULL)
    {
        printf("Error: Cannot open marks.txt\n");
        return 1;
    }

    yyin = fp;

    yylex();

    fclose(fp);

    printf("\n------ Student Marks Report ------\n");
    printf("Total Records : %d\n", total);
    printf("Valid Records : %d\n", valid);
    printf("Invalid Records: %d\n", invalid);
    printf("Passed : %d\n", passed);
    printf("Failed : %d\n", failed);

    return 0;
}

Writing marks.l


In [15]:
!flex marks.l
!gcc lex.yy.c -o marks
!./marks

Valid Record - PASS : 101 CS 85
Valid Record - FAIL : 102 MA 35
Invalid Record : 10 CS 90
Invalid Record (Marks out of range) : 103 EE 120

------ Student Marks Report ------
Total Records : 4
Valid Records : 2
Invalid Records: 2
Passed : 1
Failed : 1
